* Simple LLM calls with streaming
* Dynamic prompt templates (translation app)
* Building chains (story generatoe with analysis)
* Conversational Q&A Assistant with memory
* tool integration (calculator and weather)

In [ ]:
import langchain


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [5]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Example 1: Simple LLM call with Streaming

In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [10]:
model = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.7)

In [11]:
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002016C026F90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002016C1A8E10>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
## create messages
messages = [
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("What are the top 2 benefits of using Langchain?")
]

In [13]:
## invoke the model
response = model.invoke(messages)

In [16]:
print(response.content)

Langchain is a powerful AI tool that allows users to create complex, multi-step conversations with AI models. Some of the top benefits of using Langchain include:

1. **Improved Conversation Flow**: Langchain's ability to create multi-step conversations enables users to build more sophisticated and natural-sounding conversations with AI models. This can be particularly useful for tasks such as chatbots, virtual assistants, and language translation.

2. **Increased Productivity**: Langchain allows users to automate and streamline tasks that involve conversing with AI models, freeing up time and resources for more complex and creative work. This can be particularly useful for tasks such as research, data analysis, and content generation.

It's worth noting that Langchain is still an emerging technology and its benefits may expand as the technology continues to evolve.


In [17]:
## streaming example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

Langchain is an open-source framework for building large language models and multimodal models. Its primary goal is to simplify the development of AI models by providing an interface to connect different models and data sources. 

The top 2 benefits of using Langchain are:

1. **Simplified Model Integration**: Langchain enables developers to easily integrate various AI models, including large language models like LLaMA and other multimodal models, into a single application. This integration makes it simpler to develop complex AI-powered systems that can process and generate human-like text, images, and other forms of media.

2. **Unified API and Data Access**: Langchain provides a unified API for accessing different models and data sources, making it easier to retrieve and process information from various sources. This unified interface simplifies the development process by reducing the complexity of working with multiple APIs and data sources, allowing developers to focus on building 

### Dynamic Prompt Templates

In [22]:
from langchain_core.prompts import ChatPromptTemplate

### translation APP

translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator that translates from {source_lang} to {target_lang}. Maintain the tone and style of the original text."),
    ("user", "{text}")
])

prompt = translation_template.invoke({
    "source_lang": "English",
    "target_lang": "French",
    "text": "Langchain is a powerful framework for building applications with language models."
})

In [23]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator that translates from English to French. Maintain the tone and style of the original text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain is a powerful framework for building applications with language models.', additional_kwargs={}, response_metadata={})])

In [24]:
translated_response = model.invoke(prompt)
print(translated_response.content)

Langchain est un cadre puissant pour construire des applications avec des modèles de langage.


### Building your first Chain

In [32]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def create_story_chain():
    ## Template for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Generate a short and engaging story based on the following theme: {theme}. The story should be engaging and suitable for all audiences."),
        ("user", "Theme: {theme}\n main character: {character}\n setting: {setting}")
    ])

    ## Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert story analyst. Analyze the following story and provide insights on its themes, characters, and setting."),
        ("user", "{story}")
    ])

    story_chain = (
        story_prompt | model | StrOutputParser()
    )

    def analyze_story(story_text):
        return {"story": story_text}

    analysis_chain = (
        story_chain | RunnableLambda(analyze_story) | analysis_prompt | model | StrOutputParser()
    )

    return analysis_chain

In [33]:
chain = create_story_chain()

In [34]:
result = chain.invoke({
    "theme": "adventure and friendship",
    "character": "a young explorer named Alex",
    "setting": "a mysterious island filled with hidden treasures"
})
print("Story and Analysis:")
print(result)

Story and Analysis:
**Story Analysis: The Island of Azura**

**Themes:**

1. **The Power of Friendship**: The story emphasizes the importance of forming strong bonds with others, highlighting the connections that Alex and his friends make while exploring the Island of Azura. Their shared experiences and challenges strengthen their relationships, ultimately making the treasure of friendship more valuable than any material riches.
2. **Self-Discovery**: Alex's journey is a metaphor for self-discovery. As he ventures into the unknown, he learns about himself, his abilities, and his values. He discovers that the true treasure is not the gold or jewels but the friendships he has formed and the knowledge he has gained.
3. **Adventure and Exploration**: The story celebrates the thrill of adventure and the value of exploration. Alex's determination to follow in his grandfather's footsteps drives the plot, showcasing the allure of the unknown and the satisfaction of uncovering its secrets.

**C